# FinBERT Sentiment Scoring


In [1]:
%pip install ipywidgets
%pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 8.6 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm.notebook import tqdm

tqdm.pandas()


In [3]:
tweets_df = pd.read_parquet("../data/dataset/stock_tweets_sentiment_emotion_stanceScore_nomerge.parquet")

tweets_df


,ticker,text,created_at,user_id,date,sentiment,emotion_anger,emotion_disgust,emotion_fear,emotion_joy,...,emotion_sadness_pct,emotion_surprize_pct,positive_emotion,negative_emotion,uncertainty_emotion,positive_emotion_pct,negative_emotion_pct,uncertainty_emotion_pct,stance_label,stance_score
0,AAPL,summary of yesterdays webcast featuring wynn g...,2013-12-31 23:10:08+00:00,1864753100,2013-12-31,4,0.677257,0.148463,0.143604,0.021908,...,0.214632,0.197516,0.021908,0.829146,0.144724,0.341360,1.168543,0.958575,Positive,0.742110
1,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 01:18:36+00:00,1937591882,2014-01-01,4,0.677257,0.148463,0.143604,0.021908,...,0.214632,0.197516,0.021908,0.829146,0.144724,0.341360,1.168543,0.958575,Positive,0.742110
2,AAPL,iphone users are more intelligent than samsung...,2014-01-01 01:52:31+00:00,23954327,2014-01-01,5,0.651749,0.290809,0.032565,0.017082,...,0.070304,0.430871,0.017082,0.944506,0.034343,0.243422,1.417922,0.611794,Positive,0.998410
3,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 03:29:29+00:00,1933063572,2014-01-01,4,0.677257,0.148463,0.143604,0.021908,...,0.214632,0.197516,0.021908,0.829146,0.144724,0.341360,1.168543,0.958575,Positive,0.742110
4,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 03:59:03+00:00,1938270918,2014-01-01,4,0.677257,0.148463,0.143604,0.021908,...,0.214632,0.197516,0.021908,0.829146,0.144724,0.341360,1.168543,0.958575,Positive,0.742110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106333,XOM,t active morning movers at t nyse t exxon mobil …,2015-12-28 17:15:13+00:00,2342763212,2015-12-28,5,0.793153,0.089309,0.077659,0.018933,...,0.514482,0.468055,0.018933,0.889814,0.079564,0.286050,1.472964,0.938978,Positive,0.803462
106334,XOM,divest from stopcommoncore optout because chil...,2015-12-28 19:39:46+00:00,4399710563,2015-12-28,1,0.763934,0.115369,0.071567,0.021422,...,0.602386,0.859171,0.021422,0.888479,0.077954,0.333601,1.595991,1.298266,Negative,0.949051
106335,XOM,zsl stock forum zsl gold uslv zsl investing na...,2015-12-29 16:52:36+00:00,2181314366,2015-12-29,5,0.488881,0.242657,0.201586,0.045692,...,0.513626,0.298238,0.045692,0.738872,0.202946,0.646072,1.406148,1.224172,Positive,0.997038
106336,XOM,nptn recent news updated tuesday december pm g...,2015-12-29 19:03:17+00:00,2197054086,2015-12-29,1,0.456476,0.355527,0.112046,0.067945,...,0.193402,0.154526,0.067945,0.815209,0.113054,0.791015,1.324451,0.799488,Positive,0.997328


In [4]:
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    top_k=None,
    device=device
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
def run_finbert(texts, batch_size=32):
    results = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts.iloc[i:i + batch_size].tolist()
        outputs = classifier(batch, truncation=True)
        results.extend(outputs)
    return results


def parse_finbert_output(output):
    score_by_label = {d['label'].lower(): d['score'] for d in output}
    label = max(score_by_label, key=score_by_label.get)
    score = score_by_label[label]
    return (
        label,
        score,
        score_by_label.get('positive', float('nan')),
        score_by_label.get('negative', float('nan')),
        score_by_label.get('neutral', float('nan')),
    )


finbert_outputs = run_finbert(tweets_df['text'])
parsed = [parse_finbert_output(o) for o in finbert_outputs]
tweets_df[[
    'finbert_label', 'finbert_score',
    'finbert_up', 'finbert_down', 'finbert_neutral'
]] = pd.DataFrame(parsed, index=tweets_df.index)


  0%|          | 0/3324 [00:00<?, ?it/s]

In [6]:
tweets_df


,ticker,text,created_at,user_id,date,sentiment,emotion_anger,emotion_disgust,emotion_fear,emotion_joy,...,positive_emotion_pct,negative_emotion_pct,uncertainty_emotion_pct,stance_label,stance_score,finbert_label,finbert_score,finbert_up,finbert_down,finbert_neutral
0,AAPL,summary of yesterdays webcast featuring wynn g...,2013-12-31 23:10:08+00:00,1864753100,2013-12-31,4,0.677257,0.148463,0.143604,0.021908,...,0.341360,1.168543,0.958575,Positive,0.742110,neutral,0.941729,0.026364,0.031907,0.941729
1,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 01:18:36+00:00,1937591882,2014-01-01,4,0.677257,0.148463,0.143604,0.021908,...,0.341360,1.168543,0.958575,Positive,0.742110,neutral,0.941729,0.026364,0.031907,0.941729
2,AAPL,iphone users are more intelligent than samsung...,2014-01-01 01:52:31+00:00,23954327,2014-01-01,5,0.651749,0.290809,0.032565,0.017082,...,0.243422,1.417922,0.611794,Positive,0.998410,neutral,0.704061,0.285699,0.010240,0.704061
3,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 03:29:29+00:00,1933063572,2014-01-01,4,0.677257,0.148463,0.143604,0.021908,...,0.341360,1.168543,0.958575,Positive,0.742110,neutral,0.941729,0.026364,0.031907,0.941729
4,AAPL,summary of yesterdays webcast featuring wynn g...,2014-01-01 03:59:03+00:00,1938270918,2014-01-01,4,0.677257,0.148463,0.143604,0.021908,...,0.341360,1.168543,0.958575,Positive,0.742110,neutral,0.941729,0.026364,0.031907,0.941729
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106333,XOM,t active morning movers at t nyse t exxon mobil …,2015-12-28 17:15:13+00:00,2342763212,2015-12-28,5,0.793153,0.089309,0.077659,0.018933,...,0.286050,1.472964,0.938978,Positive,0.803462,neutral,0.930226,0.041858,0.027916,0.930226
106334,XOM,divest from stopcommoncore optout because chil...,2015-12-28 19:39:46+00:00,4399710563,2015-12-28,1,0.763934,0.115369,0.071567,0.021422,...,0.333601,1.595991,1.298266,Negative,0.949051,neutral,0.600150,0.023301,0.376548,0.600150
106335,XOM,zsl stock forum zsl gold uslv zsl investing na...,2015-12-29 16:52:36+00:00,2181314366,2015-12-29,5,0.488881,0.242657,0.201586,0.045692,...,0.646072,1.406148,1.224172,Positive,0.997038,neutral,0.938354,0.036500,0.025146,0.938354
106336,XOM,nptn recent news updated tuesday december pm g...,2015-12-29 19:03:17+00:00,2197054086,2015-12-29,1,0.456476,0.355527,0.112046,0.067945,...,0.791015,1.324451,0.799488,Positive,0.997328,neutral,0.899062,0.023730,0.077208,0.899062


In [ ]:
tweets_df.to_parquet("../data/dataset/stock_tweets_sentiment_emotion_stanceScore_finbert_nomerge.parquet", index=False)